# ETL - Limpieza de Resultados

Funciones de limpieza para `resultados_clase.csv` (dataset con datos intencionalmente sucios).

In [ ]:
import re
import unicodedata
import numpy as np
import pandas as pd


In [ ]:
def strip_accents(text: str) -> str:
    if text is None:
        return ''
    text = str(text)
    nfkd = unicodedata.normalize('NFKD', text)
    return ''.join(ch for ch in nfkd if not unicodedata.combining(ch))

def normalize_whitespace(text: str) -> str:
    return re.sub(r'\s+', ' ', str(text or '')).strip()

def normalize_name(name: str) -> str:
    name = normalize_whitespace(name)
    if not name:
        return np.nan
    if ',' in name:
        parts = [p.strip() for p in name.split(',', 1)]
        if len(parts) == 2 and parts[0] and parts[1]:
            name = f'{parts[1]} {parts[0]}'
    name = strip_accents(name).lower()
    name = re.sub(r'[^a-z\s]', ' ', name)
    name = normalize_whitespace(name)
    if not name:
        return np.nan
    return name.title()

def parse_distance_km(value) -> float:
    if pd.isna(value):
        return np.nan
    raw = strip_accents(normalize_whitespace(str(value))).lower()
    if raw in {'', 'nan', 'none', 'n/a', 's/d'}:
        return np.nan
    words = {
        'uno': 1.0, 'dos': 2.0, 'tres': 3.0, 'cuatro': 4.0,
        'cinco': 5.0, 'diez': 10.0, 'veintiuno': 21.0, 'maraton': 42.0
    }
    if raw in words:
        return words[raw]
    compact = raw.replace('km', '').replace('k', '')
    compact = compact.replace('-', '').strip().replace(',', '.')
    m = re.search(r'-?\d+(?:\.\d+)?', compact)
    if not m:
        if 'diez' in raw:
            return 10.0
        if 'sprint' in raw or 'kids' in raw:
            return 1.0
        return np.nan
    return float(m.group())

def canonical_modalidad(value) -> str:
    km = parse_distance_km(value)
    if pd.isna(km) or km <= 0:
        return np.nan
    km_int = int(km) if float(km).is_integer() else km
    return f'{km_int}K'

def parse_time_to_seconds(value) -> float:
    if pd.isna(value):
        return np.nan
    s = normalize_whitespace(str(value))
    if s in {'', 'nan', 'None'}:
        return np.nan
    sign = -1 if s.startswith('-') else 1
    s = s.lstrip('+-')
    m = re.fullmatch(r'(\d{1,4}):(\d{1,2}):(\d{1,2})', s)
    if not m:
        return np.nan
    hh, mm, ss = map(int, m.groups())
    if mm >= 60 or ss >= 60:
        return np.nan
    return float(sign * (hh * 3600 + mm * 60 + ss))

def format_seconds_to_hms(total_seconds) -> str:
    if pd.isna(total_seconds):
        return np.nan
    total_seconds = int(round(float(total_seconds)))
    sign = '-' if total_seconds < 0 else ''
    total_seconds = abs(total_seconds)
    hh = total_seconds // 3600
    mm = (total_seconds % 3600) // 60
    ss = total_seconds % 60
    return f'{sign}{hh:02d}:{mm:02d}:{ss:02d}'


In [ ]:
def clean_results_df(df: pd.DataFrame, max_time_hours: int = 8):
    work = df.copy()
    rename_map = {
        'Número': 'numero', 'numero': 'numero',
        'Nombre': 'nombre', 'nombre': 'nombre',
        'Modalidad': 'modalidad', 'modalidad': 'modalidad',
        'KM': 'km', 'km': 'km',
        'Tiempo': 'tiempo', 'tiempo': 'tiempo',
    }
    work = work.rename(columns=rename_map)
    required = ['numero', 'nombre', 'modalidad', 'km', 'tiempo']
    missing = [c for c in required if c not in work.columns]
    if missing:
        raise ValueError(f'Faltan columnas requeridas: {missing}')

    work['nombre_clean'] = work['nombre'].apply(normalize_name)
    work['numero_clean'] = pd.to_numeric(work['numero'], errors='coerce')
    km_from_modalidad = work['modalidad'].apply(parse_distance_km)
    km_from_km = work['km'].apply(parse_distance_km)
    work['km_clean'] = km_from_km.fillna(km_from_modalidad)
    work['modalidad_clean'] = work['km_clean'].apply(canonical_modalidad)
    work['tiempo_seconds'] = work['tiempo'].apply(parse_time_to_seconds)

    max_secs = max_time_hours * 3600
    invalid_nombre = work['nombre_clean'].isna()
    invalid_numero = work['numero_clean'].isna()
    invalid_km = work['km_clean'].isna() | (work['km_clean'] <= 0) | (work['km_clean'] > 100)
    invalid_time = work['tiempo_seconds'].isna() | (work['tiempo_seconds'] <= 0) | (work['tiempo_seconds'] > max_secs)

    work['reject_reason'] = np.nan
    work.loc[invalid_nombre, 'reject_reason'] = 'nombre_invalido'
    work.loc[invalid_numero & work['reject_reason'].isna(), 'reject_reason'] = 'numero_invalido'
    work.loc[invalid_km & work['reject_reason'].isna(), 'reject_reason'] = 'distancia_invalida'
    work.loc[invalid_time & work['reject_reason'].isna(), 'reject_reason'] = 'tiempo_invalido'

    df_rejects = work[work['reject_reason'].notna()].copy()
    df_clean = work[work['reject_reason'].isna()].copy()
    df_clean = df_clean.sort_values(['tiempo_seconds', 'numero_clean'], ascending=[True, True])
    df_clean = df_clean.drop_duplicates(subset=['numero_clean', 'nombre_clean', 'km_clean'], keep='first')

    df_clean['numero'] = df_clean['numero_clean'].astype('Int64')
    df_clean['nombre'] = df_clean['nombre_clean']
    df_clean['km'] = df_clean['km_clean'].round(2)
    df_clean['modalidad'] = df_clean['modalidad_clean']
    df_clean['tiempo'] = df_clean['tiempo_seconds'].apply(format_seconds_to_hms)
    cols_final = ['numero', 'nombre', 'modalidad', 'km', 'tiempo', 'tiempo_seconds']
    df_clean = df_clean[cols_final].reset_index(drop=True)
    return df_clean, df_rejects


In [ ]:
df_results = pd.read_csv('resultados_clase.csv', skiprows=1)
df_results.head()


In [ ]:
df_clean, df_rejects = clean_results_df(df_results, max_time_hours=8)
print('Filas entrada:', len(df_results))
print('Filas limpias:', len(df_clean))
print('Filas rechazadas:', len(df_rejects))
print('\nRechazos por motivo:')
print(df_rejects['reject_reason'].value_counts(dropna=False))


In [ ]:
df_clean.to_csv('resultados_clase_clean.csv', index=False)
df_rejects.to_csv('resultados_clase_rejects.csv', index=False)
df_clean.head()
